# Refusal-direction reproduction setup on Colab Pro

This notebook mirrors the project tasks for the first reproduction pass:

1. Verify a CUDA GPU runtime.
2. Clone/sync the project and install dependencies with `uv`.
3. Authenticate to Hugging Face for gated Llama 3.2 access.
4. Download/cache the Llama 3.2 Instruct models.
5. Prepare AdvBench harmful instructions and length-matched Alpaca benign instructions.
6. Verify the prompts use the **actual Llama 3.2 chat template**.
7. Run a short smoke test on a few prepared dataset prompts.
8. Collect residual-stream activations and plot harmful/benign divergence by layer.

Before running, choose **Runtime → Change runtime type → Hardware accelerator → GPU**.

## Configuration

Adjust these values before running the notebook if needed. The default dataset size matches the project task (`mise prepare-datasets`).

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/cayblood/refusal_direction_extended.git"
PROJECT_DIR = Path("/content/refusal_direction_extended")

MODEL_1B = "meta-llama/Llama-3.2-1B-Instruct"
MODEL_3B = "meta-llama/Llama-3.2-3B-Instruct"
DATASET_DIR = Path("data/refusal_datasets")
DATASET_SAMPLE_SIZE = 256
DATASET_SEED = 0

# Smoke test settings. Keep these small for quick validation.
SMOKE_MODEL = MODEL_1B
SMOKE_EXAMPLES_PER_CLASS = 2
SMOKE_MAX_NEW_TOKENS = 48

## Verify GPU runtime

If this cell fails, switch the runtime to a GPU runtime before continuing.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "CUDA is unavailable. In Colab, choose Runtime → Change runtime type → GPU."
)
print(torch.cuda.get_device_name(0))

## Get the project code

If you opened this notebook from the repository in Colab, the checkout may already exist. Otherwise, this clones `REPO_URL`.

In [ ]:
if PROJECT_DIR.exists():
    print(f"Using existing checkout: {PROJECT_DIR}")
    %cd {PROJECT_DIR}
    !git pull --ff-only
else:
    if not REPO_URL:
        raise ValueError(
            "Set REPO_URL to your repository URL, then rerun this cell."
        )
    !git clone {REPO_URL} {PROJECT_DIR}
    %cd {PROJECT_DIR}

## Install project tooling and dependencies

Colab runtimes are ephemeral, so this installs `uv` and syncs the project dependencies into the runtime.

In [ ]:
!pip install -q uv
!uv sync
# Install this project into the environment so the scripts' `from lib...`
# imports resolve (the library lives in src/lib). `uv sync` usually does
# this, but we install it editable explicitly to be safe on fresh Colab
# runtimes.
!uv pip install -e . --quiet
!uv run python -c "import lib; print('lib package ready')"

## Hugging Face authentication

Llama 3.2 Instruct is gated. AdvBench may also require accepting the dataset agreement. This cell prompts securely and stores the token only in the current Colab runtime environment. Do not paste tokens directly into saved notebook cells.

In [ ]:
import os
import subprocess
from getpass import getpass

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("Hugging Face token: ")

subprocess.run(
    ["uv", "run", "hf", "auth", "login", "--token", os.environ["HF_TOKEN"]],
    check=True,
)
subprocess.run(["uv", "run", "hf", "auth", "whoami"], check=True)

## Verify project environment

This uses `uv run` so it checks the project environment, not only the notebook kernel.

In [ ]:
import subprocess
import textwrap

code = textwrap.dedent(
    """
    from importlib.metadata import version

    import torch

    print("torch", version("torch"))
    print("transformer-lens", version("transformer-lens"))
    print("transformers", version("transformers"))
    print("datasets", version("datasets"))
    print("cuda", torch.cuda.is_available())
    print("gpu", torch.cuda.get_device_name(0))
    """
)

subprocess.run(["uv", "run", "python", "-c", code], check=True)

## Download/cache models

This downloads both Llama 3.2 Instruct models into the active Colab runtime cache. If you want persistent caching across Colab sessions, mount Drive and set `HF_HOME` before running this cell.

In [ ]:
!uv run hf download {MODEL_1B} --repo-type model
!uv run hf download {MODEL_3B} --repo-type model

## Prepare reproduction datasets

This mirrors:

```sh
mise prepare-datasets
```

It prepares ~256 harmful AdvBench instructions and ~256 length-matched benign Alpaca instructions, formatted with the Llama 3.2 chat template.

In [ ]:
!uv run python scripts/01_prepare_datasets.py \
  --model {MODEL_1B} \
  --sample-size {DATASET_SAMPLE_SIZE} \
  --seed {DATASET_SEED} \
  --output-dir {DATASET_DIR}

## Inspect and validate formatted prompts

This verifies that every prepared prompt has the expected Llama 3.2 chat-template structure and includes the assistant generation prompt. Wrong chat templates produce wrong activation positions, so this check is intentionally strict.

In [ ]:
import json
from pathlib import Path

paths = [DATASET_DIR / "harmful.jsonl", DATASET_DIR / "benign.jsonl"]
for path in paths:
    rows = [json.loads(line) for line in path.read_text().splitlines()]
    bad = []
    for index, record in enumerate(rows):
        prompt = record["formatted_prompt"]
        expected_prefix = (
            "<|begin_of_text|><|start_header_id|>system<|end_header_id|>"
        )
        expected_suffix = "<|start_header_id|>assistant<|end_header_id|>\n\n"
        checks = [
            prompt.startswith(expected_prefix),
            "<|start_header_id|>user<|end_header_id|>" in prompt,
            prompt.endswith(expected_suffix),
            record["instruction"] in prompt,
        ]
        if not all(checks):
            bad.append((index, checks))
    print(path, "rows=", len(rows), "bad=", len(bad))
    assert not bad, bad[:5]

first_path = DATASET_DIR / "harmful.jsonl"
first = json.loads(first_path.read_text().splitlines()[0])
print("First harmful raw instruction:")
print(first["instruction"])
print("\nFirst harmful formatted prompt prefix:")
print(repr(first["formatted_prompt"][:500]))

## Dataset smoke test on a few prepared inputs

This loads the smaller Llama 3.2 1B Instruct model with TransformerLens and generates short completions from the already-formatted dataset prompts. This validates the prepared files, tokenizer formatting, model loading, and generation path without running the full experiment.

In [ ]:
!uv run python scripts/02_smoke_datasets.py \
  --dataset-dir {DATASET_DIR} \
  --model {SMOKE_MODEL} \
  --examples-per-class {SMOKE_EXAMPLES_PER_CLASS} \
  --max-new-tokens {SMOKE_MAX_NEW_TOKENS} \
  --device cuda

## Optional: run the original baseline sanity prompts

This mirrors:

```sh
mise baseline
```

It uses the hard-coded harmful/benign sanity prompts in `scripts/03_baseline.py`, not the prepared dataset files.

In [ ]:
# Full default baseline over both models (active by default).
!uv run python scripts/03_baseline.py --device cuda

# Faster variant: only the smaller model and shorter completions.
# !uv run python scripts/03_baseline.py --device cuda \
#   --model {MODEL_1B} \
#   --max-new-tokens 64

## Collect residual-stream activations

This runs every prepared harmful/benign prompt through each model and captures
the residual stream (`resid_post`) at the last instruction-token position for
every layer. It saves a `[n_prompts, n_layers, d_model]` tensor per model under
`data/activations/<model>/` (kept for the direction-extraction step) and writes
a per-layer divergence summary and plot to `artifacts/activations/<model>/`.

The printed "Last-token position check" decodes the final tokens of the first
prompt so you can confirm the captured position is the post-instruction token
(`...assistant<|end_header_id|>\n\n`) and not a token inside the instruction.
This mirrors:

```sh
mise collect-activations
```

In [ ]:
!uv run python scripts/04_collect_activations.py --device cuda

# Display the sanity plots inline.
from IPython.display import Image, display

for model in [MODEL_1B, MODEL_3B]:
    slug = model.split("/")[-1]
    plot_path = Path("artifacts/activations") / slug / "divergence.png"
    if plot_path.exists():
        print(slug)
        display(Image(filename=str(plot_path)))

## Extract refusal directions

This mirrors:

```sh
mise extract-directions
```

Reads the cached activations and computes, for every captured `(position, layer)` pair, the unit-normalized difference-in-means refusal direction on the **train** split, saving `directions.pt` per model plus `directions_summary.json`. Held-out val/test prompts are reserved for the ablation sweep and the quantitative eval, so no later step is evaluated on prompts that defined the direction. This step is CPU-only but runs fine in the GPU runtime.

In [ ]:
!uv run python scripts/05_extract_directions.py

## Validate directions by ablation

This mirrors:

```sh
mise evaluate-ablation
```

For each candidate direction, project it out of the residual stream at **every** layer and measure the harmful-refusal *bypass rate* on held-out (val) prompts. The sweep ranks candidates, deep-evaluates the top few, and saves `ablation_sweep.json`, `ablation_best.json`, and a sweep plot. Note that the most causally effective layer is typically *earlier* than the layer of maximum representational separation in the divergence plot.

In [ ]:
!uv run python scripts/06_evaluate_ablation.py --device cuda
!uv run python scripts/07_plot_ablation.py

import json
from pathlib import Path

from IPython.display import Image, display

for model in [MODEL_1B, MODEL_3B]:
    slug = model.split("/")[-1]
    best = json.loads(
        (
            Path("artifacts/activations") / slug / "ablation_best.json"
        ).read_text()
    )
    b = best["best"]
    print(
        f"{slug}: best offset {b['position_offset']:+d} layer {b['layer']}; "
        f"harmful refusal "
        f"{best['baseline']['harmful_refusal_rate']:.2f} -> "
        f"{b['harmful_refusal_rate']:.2f}"
    )
    plot_path = Path("artifacts/activations") / slug / "ablation_sweep.png"
    if plot_path.exists():
        display(Image(filename=str(plot_path)))

## Quantitative ablation + addition 2x2

This mirrors:

```sh
mise evaluate-quantitative
```

On the held-out **test** split, measure four refusal rates: harmful baseline vs ablated (necessity), and benign baseline vs the direction added (sufficiency). `alpha` is in units of the raw difference-in-means norm; a short sweep picks the smallest alpha that pushes benign refusal past a threshold. Watch for the over-steering inverted-U: benign refusal peaks around alpha 1-2 and then collapses as large additions corrupt the residual stream into incoherent text.

In [ ]:
!uv run python scripts/08_evaluate_quantitative.py --device cuda
!uv run python scripts/09_plot_quantitative.py

import json
from pathlib import Path

from IPython.display import Image, display

for model in [MODEL_1B, MODEL_3B]:
    slug = model.split("/")[-1]
    q = json.loads(
        (
            Path("artifacts/activations") / slug / "quantitative_2x2.json"
        ).read_text()
    )
    t = q["table"]
    bl, iv = t["baseline"], t["intervention"]
    print(f"\n{slug} (addition alpha={q['chosen_alpha']}):")
    print("                harmful   benign")
    print(
        f"  baseline       {bl['harmful_refusal_rate']:.2f}     "
        f"{bl['benign_refusal_rate']:.2f}"
    )
    print(
        f"  intervention   {iv['harmful_refusal_rate_ablated']:.2f}"
        f"     {iv['benign_refusal_rate_added']:.2f}"
    )
    plot_path = Path("artifacts/activations") / slug / "quantitative_2x2.png"
    if plot_path.exists():
        display(Image(filename=str(plot_path)))

## Cross-scale transfer of the refusal direction

This mirrors:

```sh
mise evaluate-transfer
```

Fit a linear map between the 1B and 3B activation spaces (2048 -> 3072) on the **val** split, push the 1B refusal direction through it, and test whether the transferred direction ablates refusal in the 3B as well as the native direction. A random-direction control is included. Caveat: fitting on val (disjoint prompts, but the *same* harmful/benign distribution) gives a high cosine; the independent-distribution control in the next section is the honest test.

In [ ]:
!uv run python scripts/11_evaluate_transfer.py --device cuda

import json
from pathlib import Path

slug = MODEL_3B.split("/")[-1]
d = json.loads(
    (Path("artifacts/activations") / slug / "transfer_summary.json").read_text()
)
print("cos(transferred, native)       =", d["cos_transfer_vs_native"])
print("held-out reconstruction error  =", d["held_out_reconstruction_error"])
print("ablation harmful refusal       :", d["ablation_harmful_refusal_rate"])

## Bonus control: an independent fitting distribution

The transfer map above is fit on the same harmful/benign distribution that defines the direction, so a skeptic can argue the map already "saw" the refusal axis. To rule that out, refit the map on a fully independent, format-matched neutral instruction set (Dolly-15k) containing no harmful/benign prompts.

First prepare the generic instruction set. This mirrors:

```sh
mise prepare-generic
```

In [ ]:
!uv run python scripts/12_prepare_generic.py

### Collect generic activations

This mirrors:

```sh
mise collect-generic-activations
```

Runs the generic prompts through both models and saves `resid_post_generic.pt` per model, the independent fitting data for the transfer control.

In [ ]:
!uv run python scripts/13_collect_generic_activations.py --device cuda

### Independent-distribution transfer control

This mirrors:

```sh
mise evaluate-transfer-independent
```

If the transfer is real, the cosine and the ablation effect should survive refitting the map on this independent distribution. If they collapse — cosine drops and the transferred direction stops bypassing refusal (approaching the random control, which already has no effect) — then the earlier positive was an artifact of the fitting distribution, not a genuine cross-scale isomorphism.

In [ ]:
!uv run python scripts/14_evaluate_transfer_independent.py --device cuda

import json
from pathlib import Path

slug = MODEL_3B.split("/")[-1]
base = Path("artifacts/activations") / slug
rows = [
    ("val-fit (same distribution)", "transfer_summary.json"),
    ("generic-fit (independent)", "transfer_independent_summary.json"),
]
for label, fname in rows:
    d = json.loads((base / fname).read_text())
    a = d["ablation_harmful_refusal_rate"]
    print(
        f"{label}: cos={d['cos_transfer_vs_native']:.3f}  "
        f"transferred_ablation={a['transferred_1b']:.3f}  "
        f"(native={a['native_3b']:.3f}, random={a['random_control']:.3f})"
    )

## Useful variants

You can rerun dataset preparation with a smaller sample while debugging:

```sh
!uv run python scripts/01_prepare_datasets.py --sample-size 16 --output-dir data/refusal_datasets_debug
```

If AdvBench Hugging Face access is unavailable, the script automatically falls back to the canonical AdvBench CSV used by `llm-attacks`.